<a href="https://colab.research.google.com/github/ShiX27/ds2002-fa26/blob/main/notebooks/02-sql-databases/2026-09-11%20%E2%80%94%20SQL%20Challenge%20Set%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q1 = q('''
SELECT t.title, a.name, a.country
FROM tracks t
JOIN artists a
ON t.artist_id = a.artist_id
''')
q1

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


The processing order is: FROM -> JOIN -> ON -> SELECT

The order SQL processes the query is: First, start with the "tracks" table and call it "t" as a short name. Second, bring in the "artists" table and call it "a". Third, match the artist_id column in "tracks" with the artist_id column in "artists" and connect the rows with the same artist_id. After the rows have been matched, show title column from "tracks" and name and country columns from "artists".

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q2 = q('''
SELECT genre, AVG(seconds) AS avg_seconds
FROM tracks
GROUP BY genre
ORDER BY avg_seconds DESC
LIMIT 1
''')
q2

,genre,avg_seconds
0,Electronic,287.5


Electronic genre has the longest average track length (287. 5 seconds).

The processing order is: FROM -> GROUP BY -> SELECT -> ORDER BY -> LIMIT

The order SQL processes the query is: First, start with the "tracks" table. Second, put the rows with the same "genre" into the same group. Third, for each group, show the "genre" and calculate the average track length in "seconds", storing the result as "avg_seconds". Fourth, sort the groups from the highest average track length to the lowest. Finally, keep only the first row, which contains the genre with the highest avg_seconds.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q3 = q('''
SELECT user, COUNT(*) AS plays, COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY plays DESC
''')
q3

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,dan,2,2
3,cara,2,2


Ava has 4 plays and 4 distinct tracks.

Ben has 3 plays and 3 distinct tracks.

Dan has 2 plays and 2 distinct tracks.

Cara has 2 plays and 2 distinct tracks.

The processing order is: FROM -> GROUP BY -> SELECT -> ORDER BY

The order SQL processes the query is: First, start with the "plays" table. Second, put the rows with the same "user" into the same group. Third, for each group, show "user", count how many plays each user has and store the result as "plays" (new column name), and count the number of distinct track_id values for each user and store the result as "distinct_tracks" (new column name). Finally, sort the groups from the highest "plays" to the lowest (descending order).

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q4 = q('''
SELECT t.track_id, t.title
FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id
WHERE p.track_id IS NULL
''')
q4

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


Track 17, 'Ridgeline,' and Track 18, 'Unititled Demo,' have never been played.

The processing order is: FROM -> LEFT JOIN/ON -> WHERE -> SELECT

The order SQL processes the query is: First, start with the "tracks" table and call it "t" as a short name. Second, keep all rows from the "tracks" table on the left, bring in the "plays" table on the right, and call it "p". Third, match the track_id column in "tracks" with the track_id column in "plays". If a track has no matching row in "plays', all columns from "plays" come back as NULL. Then, the WHERE clause keeps only the unmatched tracks where p.track_id is NULL. Finally, show the track_id and title columns from "tracks".

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q5 = q('''
SELECT a.name, SUM(t.seconds) AS total_seconds, ROUND(SUM(t.seconds) / 60.0, 1) AS total_minutes
FROM artists a
JOIN tracks t ON a.artist_id = t.artist_id
JOIN plays p ON t.track_id = p.track_id
GROUP BY a.name
ORDER BY total_seconds DESC
''')
q5

,name,total_seconds,total_minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


The processing order is: FROM -> JOIN/ON -> JOIN/ON -> GROUP BY -> SELECT -> ORDER BY

The order SQL processes the query is: First, start with the "artists" table and call it "a" as a short name. Second, bring in the "tracks" table, call it "t", match the artist_id column in "artists" with the artist_id column in "tracks", and connect the rows with the same artist_id. Third, bring in the "plays" table, call it "p", match the track_id column in "tracks" with the track_id column in "plays", and connect the rows with the same track_id. Fourth, put the rows with the same artist "name" into the same group. Fifth, for each group, show the "name" column from "artists", calucate the total listening time by summing seconds in the "seconds" column from "tracks", and store the result as "total_seconds". Then, divide the total seconds by 60.0, round to one decimal, and store the result as "total_minutes". Finally, sort the groups from the highest "total_seconds" to the lowest.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q6 = q('''
SELECT track_id, title
FROM tracks
WHERE genre IS NULL
''')
q6
# WHERE genre != 'Pop' returns rows where genre is different from 'Pop',
# but rows where genre is NULL are excluded because comparing NULL with 'Pop' results in UNKNOWN, not TRUE.
# A WHERE clause only returns rows where the condition is TRUE, so the NULL row is excluded.

,track_id,title
0,18,Untitled Demo


Track 18, 'Untitled Demo,' is missing a genre.

The processing order is: FROM -> WHERE -> SELECT

The order SQL processes the query is: First, start with the "tracks" table. Then, the WHERE clause keeps the tracks where genre is NULL. Finally, show the track_id and title columns from "tracks".

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q7 = q('''
SELECT played_on, COUNT(*) AS plays, COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on
''')
q7

,played_on,plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


The processing order is: FROM -> GROUP BY -> SELECT -> ORDER BY

The order SQL processes the query is: First, start with the "plays" table. Second, put the rows with the same "played_on" date into the same group. Third, for each group, show "played_on" date, count plays per date and store the result as "plays" (new column name), and count distinct users per date and store the result as "distinct_users" (new column name). Finally, sort the groups from the earliest "played_on" date to the latest. (Ascending order is the default.)

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

Q4 gave me the most trouble. Since the prompt asked me to identify which tracks have never been played, I originally thought about using COUNT() to find tracks with zero plays. I did not realize at first that with a LEFT JOIN, if a track has no matching row in the "plays" table, the columns from "plays" come back as NULL. I also did not think about using a WHERE clause to filter for those unmatched rows. Once I understood how LEFT JOIN handles tracks with no matching plays, I realized that I could use WHERE p.track_id IS NULL to identify the tracks that had no matching plays.